First script: runs LightGlue to generate matches between CORONA and NAIP images.

In [ ]:
import os
import numpy as np
import torch

from lightglue import LightGlue, SuperPoint
from lightglue.utils import load_image, rbd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
extractor = SuperPoint(
    max_num_keypoints=4096
).eval().to(device)
matcher = LightGlue(
    features="superpoint"
).eval().to(device)

pairs_file = "../corona-auto-georeferencing/gcp_generation/pairs.txt"

corona_dir = "../corona-auto-georeferencing/gcp_generation/corona_patches"
naip_dir = "../corona-auto-georeferencing/gcp_generation/naip_patches"

output_dir = "outputs"
os.makedirs(output_dir, exist_ok=True)

with open(pairs_file, "r") as f:
    pairs = f.readlines()

print(f"Found {len(pairs)} image pairs")

for idx, line in enumerate(pairs):

    cor_name, naip_name = line.strip().split()

    cor_path = os.path.join(corona_dir, cor_name)
    naip_path = os.path.join(naip_dir, naip_name)

    print(f"\n[{idx+1}/{len(pairs)}] Matching:")
    print(f"  {cor_name}")
    print(f"  {naip_name}")

    image0 = load_image(cor_path).to(device)
    image1 = load_image(naip_path).to(device)

    with torch.no_grad():

        feats0 = extractor.extract(image0)
        feats1 = extractor.extract(image1)

        matches01 = matcher({
            "image0": feats0,
            "image1": feats1
        })

    feats0, feats1, matches01 = [
        rbd(x) for x in [feats0, feats1, matches01]
    ]

    matches = matches01["matches"]

    kpts0 = feats0["keypoints"][matches[:, 0]].cpu().numpy()
    kpts1 = feats1["keypoints"][matches[:, 1]].cpu().numpy()

    if "scores" in matches01:
        scores = matches01["scores"].detach().cpu().numpy()

    elif "matching_scores" in matches01:
        scores = matches01["matching_scores"].detach().cpu().numpy()

    else:
        scores = np.ones(len(matches))

    print(f"  Raw matches: {len(kpts0)}")

    out_name = (
        f"{os.path.splitext(cor_name)[0]}_"
        f"{os.path.splitext(naip_name)[0]}_matches.npz"
    )

    np.savez(
        os.path.join(output_dir, out_name),
        keypoints0=kpts0,
        keypoints1=kpts1,
        matching_scores=scores
    )

print("\nFinished matching all patch pairs")

Second script: filters matches from .npz files of the output of the first script to only maintain strong matches

In [ ]:
import numpy as np
import cv2
import os

input_dir = "outputs"
output_dir = "filtered"

os.makedirs(output_dir, exist_ok=True)

for file in os.listdir(input_dir):

    if not file.endswith(".npz"):
        continue

    data = np.load(os.path.join(input_dir, file))

    kpts0 = data["keypoints0"]
    kpts1 = data["keypoints1"]
    scores = data["matching_scores"]

    mask = scores > 0.2

    kpts0 = kpts0[mask]
    kpts1 = kpts1[mask]

    if len(kpts0) < 20:
        continue

    H, inliers = cv2.findHomography(
        kpts0,
        kpts1,
        cv2.RANSAC,
        5.0
    )

    if inliers is None:
        continue

    inliers = inliers.ravel() == 1

    kpts0 = kpts0[inliers]
    kpts1 = kpts1[inliers]

    print(file, len(kpts0))

    np.savez(
        os.path.join(output_dir, file),
        keypoints0=kpts0,
        keypoints1=kpts1
    )

Third script: generates csv of gcps from filtered matches

In [ ]:
import numpy as np
import rasterio
import os

filtered_dir = "filtered"
corona_path = "../corona-auto-georeferencing/gcp_generation/CORONAv2.tif"
naip_path = "../corona-auto-georeferencing/gcp_generation/NAIPv3.tif"

COR_W, COR_H = 1536, 1132
NAIP_W, NAIP_H = 1920, 1440

gcp_list = []

with rasterio.open(corona_path) as cor, rasterio.open(naip_path) as naip:

    for file in os.listdir(filtered_dir):
        if not file.endswith(".npz"):
            continue

        data = np.load(os.path.join(filtered_dir, file))

        kpts0 = data["keypoints0"]  # corona patch coords
        kpts1 = data["keypoints1"]  # naip patch coords

        patch_id = int(file.split("_")[1])

        patches_per_row = cor.width // COR_W
        row = (patch_id // patches_per_row) * COR_H
        col = (patch_id % patches_per_row) * COR_W

        for (cx, cy), (nx, ny) in zip(kpts0, kpts1):

            corona_x = cx + col
            corona_y = cy + row

            nx_orig = nx * (COR_W / NAIP_W)
            ny_orig = ny * (COR_H / NAIP_H)

            X, Y = naip.xy(int(ny_orig), int(nx_orig))

            gcp_list.append([corona_x, corona_y, X, Y])


import csv

with open("gcps.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["corona_x", "corona_y", "X", "Y"])
    writer.writerows(gcp_list)

print(f"Saved {len(gcp_list)} GCPs to gcps.csv")